In [2]:
file_path = "D:\\EDAPI\\EDAPI-Bench-main\\data\\EditDeprecatedAPI\\deepseek-1.3b\\all.json"          # đường dẫn file input
output_path = "D:\\EDAPI\\EDAPI-Bench-main\\data\\EditDeprecatedAPI\\deepseek-1.3b\\solve.json" # file output

In [3]:
import json
import os

# Kiểm tra thư mục output, nếu chưa có thì tạo mới
output_dir = os.path.dirname(output_path)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Đọc dữ liệu từ file JSON
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

solve_data = []

for item in data:
    probing_input = item.get("probing input", "")
    reference = item.get("reference", "")
    
    dep_api_list = item.get("deprecated api", [])
    rep_api_full = item.get("replacement api", "")
    
    # Bỏ qua mẫu nếu không có deprecated api
    if not dep_api_list:
        continue

    rep_parts = rep_api_full.split('.')

    # Tách mỗi phần tử trong list thành 1 mẫu (sample) riêng biệt
    for dep_api_full in dep_api_list:
        dep_parts = dep_api_full.split('.')
        common_parts = []
        
        # So sánh từng token từ trái sang phải để tìm phần chung
        for d, r in zip(dep_parts, rep_parts):
            if d == r:
                common_parts.append(d)
            else:
                break

        # Nếu có phần chung
        if common_parts:
            last_common_token = common_parts[-1] # Token cuối cùng của phần chung bên trái
            
            # Lấy phần còn lại
            dep_diff = ".".join(dep_parts[len(common_parts):])
            rep_diff = ".".join(rep_parts[len(common_parts):])

            # Ghép token cuối với phần còn lại
            new_dep = f"{last_common_token}.{dep_diff}" if dep_diff else last_common_token
            new_rep = f"{last_common_token}.{rep_diff}" if rep_diff else last_common_token
        else:
            # Nếu hoàn toàn không có gì chung
            new_dep = dep_api_full
            new_rep = rep_api_full
            rep_diff = rep_api_full

        # Xử lý cắt chuỗi tạo prompt cho mẫu hiện tại
        prompt = probing_input
        # Tìm phần còn lại của replacement api (rep_diff) trong chuỗi reference
        if rep_diff and rep_diff in reference:
            ref_prefix = reference.split(rep_diff)[0]
            prompt += ref_prefix

        # Tạo mẫu mới và đưa vào mảng kết quả
        new_item = {
            "prompt": prompt,
            "deprecated_api": new_dep,
            "replacement_api": new_rep
        }
        solve_data.append(new_item)

# Ghi danh sách các mẫu ra file mới
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(solve_data, f, indent=4, ensure_ascii=False)

print(f"Đã xử lý xong! File kết quả gồm {len(solve_data)} mẫu được lưu tại: {output_path}")

Đã xử lý xong! File kết quả gồm 1350 mẫu được lưu tại: D:\EDAPI\EDAPI-Bench-main\data\EditDeprecatedAPI\deepseek-1.3b\solve.json
